## Micrograd

In [783]:
import math

class Value:

    def __init__(self, data, _children = (), op = (), label = " "):
        self.data = data
        self.grad = 0
        self._backward = lambda: None 
        self._prev = set(_children)
        self._op = op
        self._label = label
    
    def __repr__(self):
        return f"Value(label={self._label}, data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        
        out._backward = _backward 
        return out

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        
        out._backward = _backward 
        return out

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * (other**-1)

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int and float powers are supported for now"
        out = Value(self.data ** other, (self,), f"**{other}")

        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad

        out._backward = _backward
        return out

    def tanh(self):
        n = self.data
        t = (math.exp(2*n) - 1)/(math.exp(2*n) + 1)
        out = Value(t, (self, ), label="tanh")

        def _backward():
            self.grad +=  (1 - t**2) * out.grad
        
        out._backward = _backward 
        return out

    def backprop(self):
        self._backward()
        print(f"label: {self._label}, grad: {self.grad}")
        for child in self._prev:
            child.backprop()

In [784]:
a = Value(3.0, label="a")
b = a + a
b.grad = 1
b.backprop()

label:  , grad: 1
label: a, grad: 2.0


In [785]:
a + 1
a * 2
2 * a
2 + a

Value(label= , data=5.0)

In [786]:
x1 = Value(2.0, label="x1")
x2 = Value(0.0, label="x2")

w1 = Value(-3.0, label="w1")
w2 = Value(1.0, label="w2")

b = Value(6.8813735870195432, label="b")

x1w1 = x1*w1
x1w1._label = "x1*w1"

x2w2 = x2*w2
x2w2._label = "x2*w2"

x1w1x2w2 = x1w1 + x2w2
x1w1x2w2._label = "w1*w1 + x2*w2"

n = x1w1x2w2 + b
n._label = "n"

o = n.tanh()
o._label = "o"

In [787]:
o.grad = 1.0

In [788]:
o.backprop()

label: o, grad: 1.0
label: n, grad: 0.4999999999999999
label: w1*w1 + x2*w2, grad: 0.4999999999999999
label: x2*w2, grad: 0.4999999999999999
label: x2, grad: 0.4999999999999999
label: w2, grad: 0.0
label: x1*w1, grad: 0.4999999999999999
label: w1, grad: 0.9999999999999998
label: x1, grad: -1.4999999999999996
label: b, grad: 0.4999999999999999


In [789]:
import random

class Neuron:

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for i in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        assert len(x) == len(self.w), "number of inputs is not equal to the number of weights"
        act = sum([xi * self.w[i] for i, xi in enumerate(x)]) + self.b
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

In [790]:
x = [2.0, 3.0]
n = Neuron(len(x))
n(x)

Value(label=tanh, data=-0.9995589470460945)

In [791]:
class Layer:

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        out = [neuron(x) for neuron in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

In [792]:
x = [2.0, 3.0]
la = Layer(len(x), 3)
la(x)

[Value(label=tanh, data=0.9082118480665184),
 Value(label=tanh, data=-0.9814604140664077),
 Value(label=tanh, data=0.06297882802546753)]

In [793]:
class MLP:

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        out = x
        for layer in self.layers:
            out = layer(out)
        return out

    def parameters(self):
        return [p for la in self.layers for p in la.parameters()]

In [794]:
x = [2.0, 3.0, -1.0]
mlp = MLP(len(x), [4, 4, 1])
mlp(x)

Value(label=tanh, data=-0.21952675318467418)

In [795]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [1.0, -1.0, -1.0, 1.0]

In [796]:
ypreds = [mlp(x) for x in xs]
ypreds

[Value(label=tanh, data=-0.21952675318467418),
 Value(label=tanh, data=0.33703393784332963),
 Value(label=tanh, data=0.11006890260431104),
 Value(label=tanh, data=0.07084239262580438)]

In [797]:
loss = sum([(ypred - y)**2 for y, ypred in zip(ys, ypreds)])
loss._label = "loss"
loss

Value(label=loss, data=5.370492080548473)

In [798]:
loss.grad = 1.0
loss.backprop()

label: loss, grad: 1.0
label:  , grad: 1.0
label:  , grad: 1.0
label:  , grad: 2.2201378052086223
label: tanh, grad: 2.2201378052086223
label:  , grad: 2.193240473104465
label:  , grad: 2.193240473104465
label:  , grad: 2.193240473104465
label:  , grad: 2.193240473104465
label:  , grad: 2.193240473104465
label:  , grad: 2.193240473104465
label: tanh, grad: 1.0950448734435057
label:  , grad: 0.018218829823537383
label:  , grad: 0.018218829823537383
label:  , grad: 0.018218829823537383
label:  , grad: 0.018218829823537383
label: tanh, grad: -0.012378185945304472
label:  , grad: -0.0023620290200307825
label:  , grad: -0.0023620290200307825
label:  , grad: -0.0023620290200307825
label:  , grad: -0.0023620290200307825
label:  , grad: -0.0019116188339292134
label:  , grad: -0.0023620290200307825
label:  , grad: -0.0023620290200307825
label:  , grad: -0.0023620290200307825
label:  , grad: -0.0023620290200307825
label:  , grad: -0.0023620290200307825
label:  , grad: 0.0008849700668950726
label

In [799]:
len(mlp.parameters())

41

In [800]:
STEP = 0.01

for p in mlp.parameters():
    p_old = p.data
    p.data += p.grad * (-STEP)
    print(f"parameter updated from {p_old} to {p.data}")

parameter updated from -0.915520193882233 to 0.09091214346563681
parameter updated from 0.4636734449809554 to 1.1999803707879648
parameter updated from 0.22446884520043442 to -0.2013792951358293
parameter updated from 0.6271619661035528 to 0.7271925142221708
parameter updated from -0.050062269449818286 to 0.22946905145541874
parameter updated from -0.19509256620965143 to -0.0033334820131866727
parameter updated from 0.1025490612030171 to 0.002131402782226996
parameter updated from 0.8469198811495053 to 0.867732094805694
parameter updated from -0.14750742176179377 to 0.10691056228881346
parameter updated from -0.5349061224939828 to -0.7629246779887389
parameter updated from 0.6126089453910881 to 0.6111952933286288
parameter updated from -0.3928063021855388 to -0.42168033148368983
parameter updated from -0.37466519648583296 to -0.5640683169366162
parameter updated from 0.8424686620088857 to 0.6173623920448805
parameter updated from 0.8093121708997042 to 0.8850824608937501
parameter updat

In [801]:
TRAINING_ITER = 10

def train(mlp, xs, ys):
    for i in range(TRAINING_ITER):
        # forward
        ypreds = [mlp(x) for x in xs]
        loss = sum([(ypred - y)**2 for y, ypred in zip(ys, ypreds)])
        loss._label = "loss"
        loss.grad = 1
        # backward
        loss.backprop()
        # update
        for p in mlp.parameters():
            p.data += p.grad * (-STEP)

        
    

In [802]:
train(mlp, xs, ys)

ypreds = [mlp(x) for x in xs]
loss = sum([(ypred - y)**2 for y, ypred in zip(ys, ypreds)])
loss

label: loss, grad: 1
label:  , grad: 1.0
label:  , grad: 1.0
label:  , grad: 2.692595439745552
label: tanh, grad: 2.692595439745552
label:  , grad: 2.3696937111102048
label:  , grad: 2.7627500330921944
label:  , grad: 2.3696937111102048
label:  , grad: 2.3696937111102048
label:  , grad: 2.3696937111102048
label:  , grad: 2.3696937111102048
label:  , grad: 2.3696937111102048
label: tanh, grad: 1.62341851231755
label:  , grad: 1.302083560458246
label:  , grad: 2.466184027383902
label:  , grad: 1.302083560458246
label:  , grad: 1.302083560458246
label:  , grad: 1.302083560458246
label:  , grad: 1.302083560458246
label:  , grad: 1.302083560458246
label:  , grad: 0.4837681059749058
label: tanh, grad: 0.48452475690425456
label:  , grad: 0.052979418953794266
label:  , grad: -9.950075392908003
label:  , grad: 0.052979418953794266
label:  , grad: 0.052979418953794266
label:  , grad: 0.052979418953794266
label:  , grad: 0.052979418953794266
label:  , grad: -100.6167440253101
label:  , grad: 0.00

Value(label= , data=0.07283844220280034)